In [113]:
from __future__ import annotations

Clasa corespunzatoare nodurilor din arborele de cautare

In [ ]:
class NodArbore:
    #constructorul
    def __init__(self, informatie = None, parinte: NodArbore = None, cost = 0):
        self.informatie = informatie
        self.parinte = parinte
        self.g = cost # costul de la rădăcină la nodul curent

    def __lt__(self, elem):
        return self.g<elem.g

    def drumRadacina(self):
        nod=self
        lDrum=[]
        while nod:
            lDrum.append(nod)
            nod=nod.parinte
        return lDrum[::-1]

    def inDrum(self,infonod):
        nod=self
        while nod:
            if nod.informatie==infonod:
                return True
            nod=nod.parinte
        return False
    
    def nodInDrum(self,infonod) -> NodArbore:
        nod=self
        while nod:
            if nod.informatie==infonod:
                return nod
            nod=nod.parinte
        return False


    def __str__(self):
        return str(self.informatie)

    #c (a->b->c)
    def __repr__(self):
        drum=self.drumRadacina()
        return f"{self.informatie}, cost:{self.g}, lg:{len(drum)-1} ({'->'.join([str(x) for x in drum])})"


Clasa corespunzatoare grafului problemei

In [115]:
class Graf:
    def __init__(self, matr=None, start=None, scopuri=None):
        self.matr=matr
        self.start=start
        self.scopuri=scopuri

    def scop(self, informatieNod):
        return informatieNod in self.scopuri

    def succesori_adiacenta(self, nod):
        lSuccesori=[]
        for infoSuccesor in range(len(self.matr)):
            if self.matr[nod.informatie][infoSuccesor]==1 and not nod.inDrum(infoSuccesor):
                lSuccesori.append(NodArbore(infoSuccesor, nod))
        return lSuccesori
    
    
Graf.succesori=Graf.succesori_adiacenta

Matricea de adiacenta:

In [116]:
m = [
    [0, 1, 0, 1, 1, 0, 0, 0, 0, 0],
    [1, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 1, 0, 1, 0, 0],
    [1, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 1, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
]



<b>Căutarea bidirecțională</b>
<p>Căutăm de la start spre scop, dar, în paralel și de la scop spre start și dacă găsim un nod de intersecție, reconstruim soluția și o returnăm</p>
<p>Pași:</p>
<ul>
    <li>Se pune nodul start într-o coadă și nodul scop în altă coadă.</li>
    <li>
    Repetitiv (pentru ambele cozi în același timp): 
        <ul>
            <li>dacă nu este coada vidă, se extrage primul nod din coadă și se verifică dacă este scop, caz în care se returnează o soluție. Dacă am ajuns la numărul de soluții dorit, ne oprim.</li>
            <li>expandăm nodul și adăugăm succesorii săi în coadă, cu condiția ca succesorii să nu fi fost deja vizitați pe ramurile din arbore corespunzătoare lor (atenție e posibil să fi fost vizitați pe o altă ramură din arbore, dar acest lucru nu contează, vizitarea nu se consideră la nivel global ci doar pe drumul curent al acelui succesor).</li>
            <li>Verificăm dacă există intersecție între cele două cozi</li>
        </ul>
    </li>
</ul>

<p>Definiti functia breadth_first care primește ca parametri un obiect de tip Graf  și un număr de soluții de afișat.</p>

In [117]:
def bidirectional_search(gr: Graf):
    
    def cauta_legatura(nodCurent, coadaAux):
        for nodInterm in coadaAux:
            if nodInterm.informatie==nodCurent.informatie:
                drumTotal=nodCurent.drumRadacina()[:-1]+nodInterm.drumRadacina()[::-1]
                return drumTotal
        return False
            
    coada=[NodArbore(gr.start)]
    coada_i=[]
    for scop in gr.scopuri:
        coada_i.append(NodArbore(scop))
    
    while coada and coada_i:
        nodCurent=coada.pop(0)
        drum=cauta_legatura(nodCurent, coada_i)
        if drum:
            print("->".join(map(str,drum)))
            return
        coada+=gr.succesori(nodCurent)
        nodCurent_i=coada_i.pop(0)
        drum_i=cauta_legatura(nodCurent_i, coada)
        if drum_i:
            print("->".join(map(str,drum_i[::-1])))
            return
        coada_i+=gr.succesori(nodCurent_i)

start = 0
scopuri = [5, 9]
gr=Graf(m,start,scopuri)
bidirectional_search(gr)

0->1->5


<b>DepthFirst limited search</b>
<p>Algoritmul depthfirst cu lungime maxima setata pentru drumul expandat</p>

In [118]:
LMAX=5
def depth_first(gr: Graf, nsol: int = 1, limita : int=LMAX):
    DF(gr, NodArbore(gr.start), nsol, limita)


def DF(gr : Graf, nodCurent: NodArbore, nsol: int, limita: int):
    if nsol <= 0 or limita <= 0:  
        return nsol

    if gr.scop(nodCurent.informatie):
        print("Solutie: ", end="")
        print(repr(nodCurent))
        nsol-=1
        if nsol==0:
            return 0

    lSuccesori =  gr.succesori(nodCurent)
    for sc in lSuccesori:
        if nsol != 0:
            nsol = DF(gr, sc, nsol, limita-1)

    return nsol

depth_first(gr, 4)

Solutie: 5, cost:0, lg:4 (0->1->2->5)
Solutie: 9, cost:0, lg:5 (0->1->2->7->9)
Solutie: 5, cost:0, lg:3 (0->1->5)
Solutie: 5, cost:0, lg:5 (0->4->7->2->5)


In [119]:
def depth_first_nerecursiv(gr : Graf, nsol: int = 1, limita : int=LMAX):
    stiva=[NodArbore(gr.start)]
    while stiva:
        nodCurent= stiva.pop(-1)
        if gr.scop(nodCurent.informatie):
            print("Solutie: ", end="")
            print(repr(nodCurent))
            nsol-=1
            if nsol==0:
                return
        if len(nodCurent.drumRadacina())<limita:
            stiva+=gr.succesori(nodCurent)[::-1]
depth_first_nerecursiv(gr, 4, LMAX)

Solutie: 5, cost:0, lg:4 (0->1->2->5)
Solutie: 9, cost:0, lg:5 (0->1->2->7->9)
Solutie: 5, cost:0, lg:3 (0->1->5)
Solutie: 5, cost:0, lg:5 (0->4->7->2->5)


In [120]:
def depth_first_iterative_deepening(gr, nsol=1):
    # vom simula o stiva prin relatia de parinte a nodului curent
    for l in range(100): #100 e lungimea de la care consider solutiile irelevante
        nsol=DFI(gr, NodArbore(gr.start), l, nsol)


def DFI(gr, nodCurent, l, nsol):
    if l < 0:
        return nsol
    if nsol <= 0:  
        return nsol

    if gr.scop(nodCurent.informatie) and l==0:
        print("Solutie: ", end="")
        print(repr(nodCurent))

        nsol-=1
        if nsol==0:
            return 0
    lSuccesori =  gr.succesori(nodCurent)
    for sc in lSuccesori:
        if nsol != 0:
            nsol = DFI(gr, sc, l-1, nsol)

    return nsol

depth_first_iterative_deepening(gr,2)

Solutie: 5, cost:0, lg:3 (0->1->5)
Solutie: 5, cost:0, lg:4 (0->1->2->5)


Pe malul unui râu se află N misionari, N canibali și o barcă cu M locuri (N și M numere naturale). Scopul e ca toți oamenii să treacă râul cu barca. Barca nu poate să plece goală. În nicio locație nu avem voie să avem mai mulți canibali decât misionari deoarece, în acest caz, canibalii îi vor ataca pe misionari.

In [121]:
f=open("input_lab2.txt","r")
continut=f.read().strip().split() # "3 2" -> ["3", "2"]
Graf.N=int(continut[0])
Graf.M=int(continut[1])
# (mis, can, mal)
start=(Graf.N, Graf.N,1)
scopuri=[  (0,0,0)  ] 

gr=Graf(start=start,scopuri=scopuri)

In [122]:
def succesori_canmis(self, nod):
        def test_condite(m,c):
            return m==0 or m>=c
        
        lSuccesori=[]
        # (mis, can, mal)
        if nod.informatie[2]==1: #malul curent (cu barca) este si cel initial
            misMalCurent=nod.informatie[0]
            canMalCurent=nod.informatie[1]
            misMalOpus=Graf.N-nod.informatie[0]
            canMalOpus=Graf.N-nod.informatie[1]       
            
        else:
            misMalCurent=Graf.N-nod.informatie[0]
            canMalCurent=Graf.N-nod.informatie[1] 
            misMalOpus=nod.informatie[0]  
            canMalOpus=nod.informatie[1]
        
        # (3,3,1)  -> (2,2,0); (3,1,0); (3,2,0)
        canMinBarca=0
        canMaxBarca=min(canMalCurent, Graf.M) 
        for cb in range(canMinBarca,canMaxBarca+1):
            if cb==0:
                misMinBarca=1
                misMaxBarca=min(misMalCurent, Graf.M) 
                misPosibiliBarca=range(misMinBarca,misMaxBarca+1)
            else:
                misMinBarca=cb
                misMaxBarca=min(misMalCurent, Graf.M-cb)
                misPosibiliBarca=[0]+list(range(misMinBarca,misMaxBarca+1))
            for mb in misPosibiliBarca:
                misMalCurentNou=misMalCurent-mb
                canMalCurentNou=canMalCurent-cb
                misMalOpusNou=misMalOpus+mb
                canMalOpusNou=canMalOpus+cb
                if not test_condite(misMalCurentNou,canMalCurentNou):
                    continue
                
                if not test_condite(misMalOpusNou,canMalOpusNou):
                    continue
                if nod.informatie[2]==1:
                    infoSuccesor=(misMalCurentNou,canMalCurentNou, 0)
                else:
                    infoSuccesor=(misMalOpusNou,canMalOpusNou,1)
                conditieNotInDrum=not nod.inDrum(infoSuccesor)
                if conditieNotInDrum:
                    nodNou=NodArbore(
                        informatie = infoSuccesor,
                        parinte = nod, 
                        cost = nod.g+1)
                    lSuccesori.append(nodNou)
        return lSuccesori
    
Graf.succesori=succesori_canmis


bidirectional_search(gr)

(3, 3, 1)->(2, 2, 0)->(3, 2, 1)->(3, 0, 0)->(3, 1, 1)->(1, 1, 0)->(2, 2, 1)->(0, 2, 0)->(0, 3, 1)->(0, 1, 0)->(1, 1, 1)->(0, 0, 0)


In [123]:
def UCS(gr, nsol):

    coada=[NodArbore(gr.start)]
    while coada:
        nodCurent=coada.pop(0)
        if gr.scop(nodCurent.informatie):
            print("Solutie: ", end="")
            print(repr(nodCurent))
            nsol-=1
            if nsol==0:
                return
        coada+=gr.succesori(nodCurent)
        coada.sort()
        
        
UCS(gr,1)

Solutie: (0, 0, 0), cost:11, lg:12 ((3, 3, 1)->(2, 2, 0)->(3, 2, 1)->(3, 0, 0)->(3, 1, 1)->(1, 1, 0)->(2, 2, 1)->(0, 2, 0)->(0, 3, 1)->(0, 1, 0)->(1, 1, 1)->(0, 0, 0))
